# SRD Honest Benchmark — Colab Runner

Runs the SRD perplexity sweep on TinyLlama-1.1B and produces the data behind `docs/SRD_RESULTS.md`.

**Pipeline:** install → unit tests → coherence smoke test → SRD perplexity sweep → K-quant baseline → plot → download.

**Hardware:** Colab T4 (free tier). Total wall-clock ~30-45 min including model download. Most cells are fast; the SRD sweep (Cell 5) is the long pole at ~10-15 min.

**Deliverables this notebook produces (downloaded in Cell 8):**
- `srd_sweep.json` — rows 1-5 of the results table (FP16 + 4 SRD configs)
- `kquant_sweep.json` — rows 6-9 (K-quant baselines, cited)
- `srd_perplexity_vs_bpw.png` — the headline plot

Paste the two JSON files back into the chat and the write-up at `docs/SRD_RESULTS.md` gets filled in.

**Source:** github.com/Orivael-Dev/axiom · plan at `/root/.claude/plans/abundant-wandering-salamander.md`

In [ ]:
# Cell 1: Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Sweep will run on CPU (~3 hours instead of ~15 min).")
    print("Runtime → Change runtime type → T4 GPU")

In [ ]:
# Cell 2: Clone the AXIOM repo + install research/quant requirements
#
# Change GIT_REF below if you want to test against a branch instead
# of main (e.g. `claude/srd-honest-prototype` before the PR merges).
import os
GIT_REF = "main"

!rm -rf axiom
!git clone --depth 1 --branch {GIT_REF} https://github.com/Orivael-Dev/axiom.git
os.chdir("axiom")
!pip install -q -r research/quant/requirements.txt
print("\n[setup] ready.")

In [ ]:
# Cell 3: Phase A — unit tests (no model download, ~2 sec)
#
# Confirms the kernel reproduces the same numbers the unit tests pin:
#   - bpw matches hand calc (4 + 8 + 32/G + 32/G)
#   - residue strictly improves MSE
#   - shape / dtype / value-range invariants hold
# If any of these fail, halt — the sweep below is meaningless until
# the kernel is right.
import os
os.environ.setdefault("AXIOM_MASTER_KEY", "colab_smoke_key_only")
!python -m pytest tests/test_axiom_quant.py -q

In [ ]:
# Cell 4: Phase B — coherence smoke test
#
# Downloads TinyLlama-1.1B (~2 GB cached after first run), applies
# SRD at alpha=1, generates 80 tokens. The output should be coherent
# English. If the model spits gibberish at alpha=1, the kernel is
# bug-bait — STOP HERE and re-check axiom_quant.py.
#
# At alpha=0 the output should be noticeably degraded but still
# English-shaped (4-bit only is lossier than the published
# Q4_K_M numbers because there's no per-block scale tuning).
!python research/quant/quantize_model.py \
    --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --alpha 1.0 \
    --prompt "Once upon a time, in a small village, " \
    --tokens 60

In [ ]:
# Cell 5: Phase C1 — SRD perplexity sweep (rows 1-5)
#
# Configs run: FP16 baseline, SRD alpha={0, 0.5, 1.0} per-block g=64,
# SRD alpha=1.0 per-tensor. Each config gets fresh weights so the
# previous run can't contaminate the next. Sliding-window PPL with
# stride=512, context=2048 (standard llama.cpp conventions).
#
# Sanity warning fires if the FP16 baseline drifts >0.5 from the
# published TinyLlama PPL (~7.7) — that means the eval harness is
# broken, not the kernel.
#
# Wall-clock: ~10-15 min on T4. The dataset fingerprint guard
# writes results/wikitext_fingerprint.txt on first run.
!python -m research.quant.bench_perplexity \
    --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --group-size 64 \
    --output research/quant/results/srd_sweep.json

In [ ]:
# Cell 6: Phase C2 — K-quant apples-to-apples comparison
#
# This cell converts TinyLlama-1.1B-Chat-v1.0 → F16 GGUF → 4 K-quant GGUFs,
# then runs llama-perplexity with --ppl-stride 512 to match the SRD eval
# exactly (same model, same stride, same context). Expected ~35 min on T4.
#
# Background: a local GTX 1660 Ti rerun using the default stride=context
# gave Q4_K_M PPL ~14.75 (vs SRD eval ~7.54 for α=0). The gap is due to
# non-overlapping chunks having no prior context — not a real quality
# difference. stride=512 closes that gap and makes the comparison citable.
#
# If this cell fails (build error, OOM, etc.), fall back to the cite-only
# path at the bottom of this cell, which takes <1 min.

import os, subprocess, sys
from pathlib import Path

LLAMA_DIR  = Path("/opt/llama.cpp")
LLAMA_BIN  = LLAMA_DIR / "build" / "bin"
GGUF_DIR   = Path("/tmp/srd_gguf")
WT2_FILE   = Path("/tmp/wikitext2_test.txt")

# ── Step 1: build llama.cpp with CUDA (≈5 min, cached on re-run) ───────────
if not (LLAMA_BIN / "llama-perplexity").exists():
    print("[build] cloning llama.cpp...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/ggml-org/llama.cpp",
                    str(LLAMA_DIR)], check=True)
    print("[build] cmake configure...")
    subprocess.run(["cmake", str(LLAMA_DIR), "-B", str(LLAMA_DIR / "build"),
                    "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release"], check=True)
    print("[build] cmake build (≈5 min)...")
    subprocess.run(["cmake", "--build", str(LLAMA_DIR / "build"),
                    "--config", "Release",
                    f"-j{os.cpu_count()}",
                    "--target", "llama-perplexity", "llama-quantize"],
                   check=True)
    print("[build] done.")
else:
    print("[build] llama-perplexity already built, skipping.")

# ── Step 2: save WikiText-2 test split to file ──────────────────────────────
if not WT2_FILE.exists():
    from datasets import load_dataset
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    WT2_FILE.write_text("\n\n".join(ds["text"]), encoding="utf-8")
    print(f"[data] saved {WT2_FILE.stat().st_size:,} bytes to {WT2_FILE}")
else:
    print(f"[data] {WT2_FILE} already exists, skipping.")

# ── Step 3: run bench_llamacpp --rerun-locally (≈35 min on T4) ─────────────
# --ppl-stride 512 matches bench_perplexity.py's sliding-window convention.
# --gguf-dir caches F16 + K-quant GGUFs so re-runs skip conversion.
!python -m research.quant.bench_llamacpp \
    --rerun-locally \
    --llama-bin {LLAMA_BIN} \
    --gguf-dir {GGUF_DIR} \
    --wikitext-file {WT2_FILE} \
    --ppl-stride 512 \
    --output research/quant/results/kquant_sweep.json

print("\n[C2] done — kquant_sweep.json written with source=rerun_local")

In [ ]:
# Cell 7: Phase E — generate the perplexity-vs-bpw plot
import os
os.makedirs("docs", exist_ok=True)
!python -m research.quant.plot_results \
    --inputs research/quant/results/srd_sweep.json,research/quant/results/kquant_sweep.json \
    --output docs/srd_perplexity_vs_bpw.png
from IPython.display import Image
Image("docs/srd_perplexity_vs_bpw.png")

In [ ]:
# Cell 8: Show the verdict + download artifacts
import json
from pathlib import Path

srd = json.load(open("research/quant/results/srd_sweep.json"))
kq  = json.load(open("research/quant/results/kquant_sweep.json"))

print("─── SRD sweep ──────────────────────────────────────")
print(f"{'name':<32} {'bpw':>6} {'PPL':>8}")
for r in srd:
    print(f"{r['name']:<32} {r['bpw_reported']:>6.2f} {r['perplexity']:>8.3f}")

print("\n─── K-quant baseline ───────────────────────────────")
print(f"{'name':<32} {'bpw':>6} {'PPL':>8}  source")
for r in kq:
    print(f"{r['name']:<32} {r['bpw_reported']:>6.2f} {r['perplexity']:>8.3f}  {r.get('source', '?')}")

print("\n─── Verdict (pre-committed decision rule) ──────────")
srd_a1 = next((r for r in srd if r['name'].startswith('srd_alpha_1.0_g')), None)
q6 = next((r for r in kq if 'Q6_K' in r['name']), None)
if srd_a1 and q6:
    margin = q6['perplexity'] - srd_a1['perplexity']
    print(f"SRD α=1.0 @ {srd_a1['bpw_reported']:.2f} bpw: PPL = {srd_a1['perplexity']:.3f}")
    print(f"Q6_K     @ {q6['bpw_reported']:.2f} bpw: PPL = {q6['perplexity']:.3f}")
    print(f"Margin (Q6_K - SRD): {margin:+.3f}")
    if margin >= 0.05:
        print("→ VERDICT: SRD beats Q6_K by ≥0.05 PPL. Worth pursuing per the plan.")
    elif margin > 0:
        print("→ VERDICT: SRD edges Q6_K but inside the 0.05 noise margin. Not worth the ~2× memory.")
    else:
        print("→ VERDICT: SRD loses to Q6_K at lower bpw. Shelve.")

# Trigger Colab download — works in Colab only.
try:
    from google.colab import files
    for path in ["research/quant/results/srd_sweep.json",
                 "research/quant/results/kquant_sweep.json",
                 "docs/srd_perplexity_vs_bpw.png"]:
        if Path(path).exists():
            files.download(path)
except ImportError:
    print("\n(google.colab not available — artifacts left at the paths above)")